# R/S benchmark — 4. NN training (global R, S, t $\to$ lambda)

Trains one `MLPRegressor` for `lambda 1` and one for `lambda 2` on the dataset stacked by
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb). Unlike the PCE stage
(`02_train_pce.ipynb`, one model per time step), this fits a single global model that also takes
$t$ as an input — query it with any $(R, S, t)$ triple, no need to pick a PCE for a specific time
step first.

`lambda 3` / `lambda 4` are **not** modelled here — read them back from the emulator dataset
directly, as scoped when this pipeline was set up.

Functions come from [`functions.py`](../functions.py): `train_and_validate_nn_lambda_benchmark`.
Prediction plots and the KL-divergence check are in
[`04_train_nn_plot.ipynb`](04_train_nn_plot.ipynb).

## 1. Libraries

In [1]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
from sklearn.model_selection import train_test_split

from functions import *

/home/casa-wand/Documentos/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples` must match [`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) — it
names the file being loaded.

In [2]:
n_latent_samples = 2500   # must match stage 3 — it is the filename prefix

feature_cols = ['r', 's', 'Time (years)']
target_cols  = ['lambda 1', 'lambda 2']

test_frac          = 0.2
hidden_layer_sizes = (64, 64)
max_iter           = 500
n_iter_no_change   = 15
random_state       = 42

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels
tick_fontsize  = 12   # font size of the tick numbers

xlim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda
ylim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda

## 3. Load the stacked dataset

In [3]:
with open(f'{n_latent_samples}_dataset_nn_benchmark.pkl', 'rb') as f:
    df_nn = dill.load(f)

print(f"Loaded {len(df_nn)} rows")
df_nn.head()

Loaded 50000 rows


,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,5.110417,2.274454,0.0,2.837573,5.559482,0.143137,0.130569
1,3.435042,2.304231,0.0,1.133046,6.011376,0.138403,0.133853
2,5.195391,1.842210,0.0,3.354314,6.465245,0.144540,0.125109
3,4.566698,2.544968,0.0,2.023816,5.183452,0.142163,0.135642
4,5.815982,2.080980,0.0,3.736439,5.682511,0.143967,0.125393


## 4. Train and validate

In [4]:
print("="*60)
print("TRAINING THE BENCHMARK NN")
print("="*60)

result = train_and_validate_nn_lambda_benchmark(
                                                   df_nn=df_nn,
                                                   feature_cols=feature_cols,
                                                   target_cols=target_cols,
                                                   test_frac=test_frac,
                                                   hidden_layer_sizes=hidden_layer_sizes,
                                                   max_iter=max_iter,
                                                   n_iter_no_change=n_iter_no_change,
                                                   random_state=random_state,
                                                   n_latent_samples=n_latent_samples,
                                                   output_dir='.',
                                                )

result['statistics']

TRAINING THE BENCHMARK NN

----------------------------------------
TRAINING NN LAMBDA MODELS
----------------------------------------
  40000 train rows, 10000 val rows


  lambda 1: R² = 0.999969, MSE = 0.00010, iterations = 19


  lambda 2: R² = 0.996641, MSE = 0.03622, iterations = 130
The NN models, scaler and validation stats have been saved!


,MSE lambda 1,R² lambda 1,MSE lambda 2,R² lambda 2
0,0.000105,0.999969,0.036225,0.996641
